# Russian River Step 1 -- download and clean raw geometry data

In this section, we choose the basin, the streams to be included in the stream-aligned mesh, and make sure that all are resolved discretely at appropriate length scales for this work.

In [ ]:
# these can be turned on for development work
%load_ext autoreload
%autoreload 2

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow
watershed_workflow.setupLogging(1)

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None
import pickle

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.sources
import watershed_workflow.sources.standard_names as names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)


## Input: Parameters and other source data

Note, this section will need to be modified for other runs of this workflow in other regions.

In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)



In [ ]:
# Set the data directory to the local space to get the locally downloaded files
print(data_dir)
watershed_workflow.utils.setDataDirectory(data_dir)


In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run

# Geometric parameters
# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = None

# working in a generic CRS
crs = watershed_workflow.crs.default_crs


In [ ]:
# set up a dictionary of source objects
#
# Data sources, also called managers, deal with downloading and parsing data files from a variety of online APIs.
sources = watershed_workflow.sources.getDefaultSources()

# log the sources that will be used here
watershed_workflow.sources.logSources(sources)


In [ ]:
# get the shape and crs of the shape
print(crs)
watershed_shapes = sources['HUC'].getShapesByID(hucs, out_crs=crs)
print(watershed_shapes.crs)

## the Watershed

In [ ]:
# Construct and plot the WW object used for storing watersheds
watershed = watershed_workflow.Watershed(watershed_shapes)
watershed.plot()

## Gage Data

In [ ]:
# find all gages in the river
nwis = watershed_workflow.sources.ManagerNWIS()
gages = nwis.getShapesByGeometry(watershed_shapes, out_crs=crs)
gages, streamflow = nwis.getStreamflow(gages, ('2000', '2026'), min_obs=365)
gages = nwis.addCOMIDs(gages)
gages = gages[~gages['comid'].isna()]  # drop sites without a reach match
gages

In [ ]:
# check that no two gages are on the same reach -- this would break our subdomain decomposition
assert len(set(gages.comid.tolist())) == len(gages)

## the Rivers 

In [ ]:
# download/collect the river network within that shape's bounds
reaches = sources['hydrography'].getShapesByGeometry(watershed.df, out_crs=crs)

# remove coastlines
reaches = reaches[reaches.ftype != 'Coastline']
print(reaches.crs)

# construct rivers
rivers = watershed_workflow.hydro.createRivers(reaches, method='hydroseq')
print(rivers[0].df.crs)

reaches

In [ ]:
#
# are all of our gage-reaches in the set of reaches?
print(sum(gages['comid'].isin(reaches['comid'])), ' of ', len(gages), ' are in the set of ALL reaches')

In [ ]:
# the outlet here needs to be modified thanks to the coastline reaches

# move the endpoint to the boundary
op = rivers[0].linestring.coords[-1]
cp = shapely.ops.nearest_points(watershed.exterior.exterior, rivers[0].linestring)[0]
print(op, cp)

rivers[0].moveCoordinate(-1, cp)

# one outlet child is a tiny offshoot of the main stem
print([len(child) for child in rivers[0].children])
rivers[0].children[0].prune()

In [ ]:
def explore():
    m = watershed.explore(style_kwds={'color':'black', 'fill':False})
    for river in rivers:
        m = river.explore(m=m, style_kwds={'color':'blue'}, name=river['name'])

    m = gages.explore(m=m, style_kwds={'color':'red', 'weight':20})
    return m

m = explore()
m

In [ ]:
# prune
rivers = watershed_workflow.reduceRivers(rivers, 
                                         remove_diversions=True,
                                         remove_braided_divergences=True)

for river in rivers:
    river.resetDataFrame()

reduced_reaches = pd.concat([r.df for r in rivers])

In [ ]:
# are all of our gages STILL in our reduced network?
print(sum(gages['comid'].isin(reduced_reaches['comid'])), ' of ', len(gages), ' are in the set of REDUCED reaches')

## Put the gages on the reaches, and use these to define subcatchments

We don't have a good way of splitting the catchment of a reach at the gage's measure of the reach.  If we had this, we could split both the reach and the catchment at the actual gage location, and make subcatchments that respect this.  Let's try to assign the gage to an upstream-most or downstream-most point on the reach instead.



In [ ]:
# add another "gage" point -- the outlet of the full domain
assert len(rivers) == 1
gages = gages.reset_index()

all_gages = gpd.GeoDataFrame(
    pd.concat([gages,
           gpd.GeoDataFrame({'comid' : rivers[0]['comid'],
                             'measure' : 0.,
                             'station_nm' : 'Russian River Outlet',
                             'ID' : 'RR-outlet',
                            },
                            index=[len(gages),],
                            geometry=[shapely.geometry.Point(rivers[0].linestring.coords[-1]),],
                            crs=gages.crs),
          ]), crs=gages.crs)

all_gages[names.NAME] = all_gages[names.ID]
all_gages.pop('index')
all_gages


In [ ]:
# Here we will try to avoid subcatchments entirely
watershed_polys = watershed.df


In [ ]:

# save these shapefiles to disk
watershed_polys.to_parquet(toOutput('watershed_polys', '01_watershed_polys.parquet'))

river_df = gpd.GeoDataFrame(pd.concat([r.to_dataframe() for r in rivers]), crs=crs)
river_df.to_parquet(toOutput('rivers', '01_rivers.parquet'))

# note -- this is not done here, but is saved from the high-res version to avoid multiple downloads
#watershed.to_parquet(toOutput('reference_watershed', '01_reference_watershed.parquet'))

streamflow.to_csv(toOutput('evaluation_discharge', '01_discharge_observations.csv'))


In [ ]:
# save output filenames
with open(toOutput('01_output_filenames', '01_output_filenames.txt'), 'wb') as fid:
    pickle.dump(output_filenames, fid)

In [ ]:
# save the location of gages
all_gages.to_parquet(toOutput('gages', '01_gages.parquet'))